# Chapter 2 — Launch configuration experiments

This notebook uses the same host setup as the vector-addition example. It changes only the kernel launch configuration and measures the average time for 100 launches. `TimingGPU` is kept in its own `.cuh` and `.cu` files.


In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov  7 14:11:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile code/cuda_check.cuh
#ifndef BOOK_CUDA_CHECK_CUH
#define BOOK_CUDA_CHECK_CUH

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define CUDA_CHECK(call) do {                                      \
    const cudaError_t status_ = (call);                            \
    if (status_ != cudaSuccess) {                                  \
        fprintf(stderr, "%s:%d: %s failed: %s\n",                \
                __FILE__, __LINE__, #call,                         \
                cudaGetErrorString(status_));                      \
        exit(EXIT_FAILURE);                                        \
    }                                                               \
} while (0)

#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef BOOK_TIMING_GPU_CUH
#define BOOK_TIMING_GPU_CUH

struct PrivateTimingGPU;

class TimingGPU {
private:
    PrivateTimingGPU *privateTimingGPU;
    void Start(unsigned int flags);
public:
    TimingGPU();
    ~TimingGPU();
    void StartCounter();
    void StartCounterFlags();
    float GetCounter();  // elapsed milliseconds
};

#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include <cuda_runtime.h>
#include "TimingGPU.cuh"
#include "cuda_check.cuh"

struct PrivateTimingGPU {
    cudaEvent_t start = nullptr;
    cudaEvent_t stop = nullptr;
    unsigned int flags = 0;
    bool events_created = false;
    bool started = false;
};

TimingGPU::TimingGPU() : privateTimingGPU(new PrivateTimingGPU{}) {}

TimingGPU::~TimingGPU() {
    if (privateTimingGPU->events_created) {
        CUDA_CHECK(cudaEventDestroy(privateTimingGPU->start));
        CUDA_CHECK(cudaEventDestroy(privateTimingGPU->stop));
    }
    delete privateTimingGPU;
}

void TimingGPU::Start(unsigned int flags) {
    if (!privateTimingGPU->events_created || privateTimingGPU->flags != flags) {
        if (privateTimingGPU->events_created) {
            CUDA_CHECK(cudaEventDestroy(privateTimingGPU->start));
            CUDA_CHECK(cudaEventDestroy(privateTimingGPU->stop));
        }
        CUDA_CHECK(cudaEventCreateWithFlags(&privateTimingGPU->start, flags));
        CUDA_CHECK(cudaEventCreateWithFlags(&privateTimingGPU->stop, flags));
        privateTimingGPU->flags = flags;
        privateTimingGPU->events_created = true;
    }
    CUDA_CHECK(cudaEventRecord(privateTimingGPU->start, 0));
    privateTimingGPU->started = true;
}

void TimingGPU::StartCounter() {
    Start(cudaEventDefault);
}

void TimingGPU::StartCounterFlags() {
    Start(cudaEventBlockingSync);
}

float TimingGPU::GetCounter() {
    if (!privateTimingGPU->started) {
        fprintf(stderr, "Call StartCounter before GetCounter.\n");
        exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(privateTimingGPU->stop, 0));
    CUDA_CHECK(cudaEventSynchronize(privateTimingGPU->stop));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds,
                                   privateTimingGPU->start,
                                   privateTimingGPU->stop));
    return milliseconds;
}


Writing code/TimingGPU.cu


In [5]:
%%writefile code/chapter02_launch_experiments.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include "TimingGPU.cuh"
#include "cuda_check.cuh"

__global__ void vector_add_grid_stride(const float *a, const float *b,
                                       float *c, int n)
{
    for (int i = blockIdx.x * blockDim.x + threadIdx.x;
         i < n; i += blockDim.x * gridDim.x) {
        c[i] = a[i] + b[i];
    }
}

int main()
{
    const int n = 100003;
    const size_t bytes = (size_t)n * sizeof(float);
    float *a = (float *)malloc(bytes);
    float *b = (float *)malloc(bytes);
    float *c = (float *)malloc(bytes);
    if (a == NULL || b == NULL || c == NULL) {
        fprintf(stderr, "Could not allocate host arrays.\n");
        return EXIT_FAILURE;
    }

    for (int i = 0; i < n; ++i) {
        a[i] = 0.25f * (i % 101);
        b[i] = 0.50f * (i % 37);
    }

    float *d_a = NULL;
    float *d_b = NULL;
    float *d_c = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    const int blocks[] = {1, 4, 32, (n + 255) / 256};
    const int threads[] = {128, 128, 128, 256};
    const int configurations = sizeof(blocks) / sizeof(blocks[0]);
    const int repeats = 100;
    TimingGPU timer;
    int failures = 0;

    for (int q = 0; q < configurations; ++q) {
        const int bcount = blocks[q];
        const int tcount = threads[q];

        // One untimed launch warms up this configuration.
        vector_add_grid_stride<<<bcount, tcount>>>(d_a, d_b, d_c, n);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());

        timer.StartCounter();
        for (int k = 0; k < repeats; ++k) {
            vector_add_grid_stride<<<bcount, tcount>>>(d_a, d_b, d_c, n);
        }
        CUDA_CHECK(cudaGetLastError());
        const float milliseconds = timer.GetCounter() / repeats;

        CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));
        int config_failures = 0;
        for (int i = 0; i < n; ++i) {
            if (c[i] != a[i] + b[i]) {
                ++config_failures;
            }
        }
        failures += config_failures;
        printf("blocks = %d, threads = %d, time = %.6f ms, validation = %s\n",
               bcount, tcount, milliseconds,
               config_failures == 0 ? "PASS" : "FAIL");
    }

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    free(a);
    free(b);
    free(c);
    return failures == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}


Writing code/chapter02_launch_experiments.cu


## Compile

The timer implementation and the host/kernel program are compiled together.


In [6]:
!nvcc -O2 -std=c++14 -arch=native -Icode code/chapter02_launch_experiments.cu code/TimingGPU.cu -o launch_experiments


## Run

Each line reports the average kernel time over 100 launches and validates the output for that launch configuration.


In [7]:
!./launch_experiments


blocks = 1, threads = 128, time = 0.219015 ms, validation = PASS
blocks = 4, threads = 128, time = 0.057016 ms, validation = PASS
blocks = 32, threads = 128, time = 0.009605 ms, validation = PASS
blocks = 391, threads = 256, time = 0.004266 ms, validation = PASS
